In [32]:

import pandas as pd
from collections import Counter
import numpy as np
import re

df = pd.read_excel("temp1.xlsx")

print(df.head())


   case_id                                  case_title  \
0  CASE_01                   State vs Chaturbhuj Singh   
1  CASE_02  Gurujlal Panika vs State of Madhya Pradesh   
2  CASE_03        Amarsingh vs State of Madhya Pradesh   
3  CASE_04        Nenavath Bujji vs State of Telangana   
4  CASE_05  Amarsingh Badri vs State of Madhya Pradesh   

                          court    year                  case_type  \
0              Delhi High Court  2013.0                   Criminal   
1  high court of madhya pradesh  2023.0                   Criminal   
2        Supreme court of India  1995.0                   Criminal   
3        Supreme court of India  2024.0  Constitutional / Criminal   
4        Supreme court of India  1995.0                   Criminal   

                                   sub_category  \
0  Culpable Homicide / Right of Private Defence   
1           Robbery / Receiving Stolen Property   
2              Murder / Circumstantial Evidence   
3             Preventive D

In [33]:
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'\n', ' ', text)

    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    text = re.sub(r'\s+', ' ', text)

    return text

In [34]:
df['clean_text'] = df['case_text'].apply(clean_text)

df[['case_text', 'clean_text']].head()

,case_text,clean_text
0,The State filed an appeal against the acquitta...,the state filed an appeal against the acquitta...
1,The case arose from criminal appeals challengi...,the case arose from criminal appeals challengi...
2,The case involved criminal appeals arising fro...,the case involved criminal appeals arising fro...
3,The appeals challenged preventive detention or...,the appeals challenged preventive detention or...
4,The case arose from criminal appeals challengi...,the case arose from criminal appeals challengi...


In [35]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2704.95it/s]


In [36]:
embeddings = model.encode(
    df['clean_text'].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 4/4 [00:02<00:00,  1.90it/s]


In [37]:
print(len(embeddings))

106


In [38]:
print(len(embeddings[0]))

384


In [39]:
print(df.shape)

(106, 14)


Similar case retrivals

In [40]:
df['combined_text'] = (

    df['case_text'].fillna('') + ' ' +

    df['legal_sections'].fillna('') + ' ' +

    df['judge_observation'].fillna('') + ' ' +

    df['legal_keywords'].fillna('') + ' ' +

    df['sub_category'].fillna('')

)

In [41]:
embeddings = model.encode(
    df['combined_text'].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.12it/s]


In [42]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype('float32')
)

print("New FAISS index created")

New FAISS index created


In [43]:
query = "The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal."

query_embedding = model.encode([query])

D, I = index.search(
    np.array(query_embedding).astype('float32'),
    k=5
)

print(I)

[[ 5 18 19 17  6]]


In [44]:
for idx in I[0]:

    print(f"""
CASE TITLE:
{df.iloc[idx]['case_title']}

CASE TYPE:
{df.iloc[idx]['case_type']}

SUB CATEGORY:
{df.iloc[idx]['sub_category']}

LEGAL SECTIONS:
{df.iloc[idx]['legal_sections']}

OUTCOME:
{df.iloc[idx]['judgment_outcome']}

CASE TEXT:
{df.iloc[idx]['case_text']}

JUDGE OBSERVATION:
{df.iloc[idx]['judge_observation']}

{"=" * 100}
""")


CASE TITLE:
Rajendra Mahto @ Kargil vs State of Jharkhand & Others

CASE TYPE:
Criminal

SUB CATEGORY:
Murder / Criminal Appeal

LEGAL SECTIONS:
IPC Section 302 and related provisions

OUTCOME:
Guilty

CASE TEXT:
The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal.

JUDGE OBSERVATION:
The High Court observed that conviction in serious criminal offences must rest upon reliable eye

suggested legal sections

In [45]:
def extract_sections(section_text):

    if pd.isna(section_text):
        return []

    text = str(section_text)

    extracted = []

    # IPC
    ipc_matches = re.findall(
        r'IPC Sections? ([0-9A-Za-z(), ]+)',
        text
    )

    for match in ipc_matches:

        nums = match.split(',')

        for num in nums:

            num = num.strip()

            if num:
                extracted.append(f"IPC {num}")

    # DV Act
    dv_matches = re.findall(
        r'Domestic Violence Act Sections? ([0-9A-Za-z(), ]+)',
        text
    )

    for match in dv_matches:

        nums = match.split(',')

        for num in nums:

            num = num.strip()

            if num:
                extracted.append(f"Domestic Violence Act {num}")

    return extracted

In [46]:
def predict_legal_sections(query, top_k=5):

    # Create embedding
    query_embedding = model.encode([query])

    # Search similar cases
    D, I = index.search(
        np.array(query_embedding).astype('float32'),
        k=top_k
    )

    weighted_sections = {}

    # Loop through retrieved cases
    for position, idx in enumerate(I[0]):

        # Distance score
        distance = D[0][position]

        # Convert distance to weight
        weight = 1 / (1 + distance)

        # Extract sections
        sections = extract_sections(
            df.iloc[idx]['legal_sections']
        )

        # Add weighted score
        for sec in sections:

            if sec not in weighted_sections:
                weighted_sections[sec] = 0

            weighted_sections[sec] += weight

    # Total weighted score
    total_weight = sum(weighted_sections.values())

    results = []

    # Convert to confidence %
    for sec, score in weighted_sections.items():

        confidence = float(round(
            (score / total_weight) * 100,
            2
        ))

        results.append(
            (sec, confidence)
        )

    # Sort descending
    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return results

In [47]:
query = """
The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal.
"""
results = predict_legal_sections(query)



print("\nSuggested Legal Sections:\n")

for sec, confidence in results:

    print(
        f"{sec}  →  {round(confidence,2)}% confidence"
    )


Suggested Legal Sections:

IPC 302  →  49.22% confidence
IPC 302 and related provisions  →  14.49% confidence
IPC 120B  →  12.62% confidence
IPC 449  →  11.84% confidence
IPC 392  →  11.84% confidence


In [48]:
from sklearn.metrics.pairwise import cosine_similarity

missing evidence


In [49]:
from collections import Counter

def detect_missing_evidence(query, top_k=5):

    # Create query embedding
    query_embedding = model.encode([query])

    # Retrieve similar cases
    D, I = index.search(
        np.array(query_embedding).astype('float32'),
        k=top_k
    )

    all_evidence = []

    # Collect evidence from similar cases
    for idx in I[0]:

        evidence_text = str(
            df.iloc[idx]["evidence_types"]
        )

        # Split evidence types
        evidence_list = evidence_text.split(',')

        for ev in evidence_list:

            ev = ev.strip().lower()

            if ev:
                all_evidence.append(ev)

    # Count evidence frequency
    evidence_counts = Counter(all_evidence)

    query_lower = query.lower()

    missing = []

    # Detect potentially missing evidence
    for evidence, count in evidence_counts.items():

        # evidence common in similar cases
        # but absent in uploaded query

        if count >= 2 and evidence not in query_lower:

            missing.append({

                "evidence": evidence
            })

    # Remove duplicates
    unique_missing = []

    seen = set()

    for item in missing:

        if item["evidence"] not in seen:

            unique_missing.append(item)

            seen.add(item["evidence"])

    return unique_missing

In [50]:
query = """
The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal.
"""

results = detect_missing_evidence(query)

results
if len(results) == 0:

    print("No major missing evidence detected.")

else:

    print("\nPossible Missing Evidence:\n")

    for item in results:

        print(f"- {item['evidence']}")


Possible Missing Evidence:

- police investigation records
- postmortem report
- recovery evidence


evidence to law mappings


In [51]:


def get_dominant_case_type(I):

    retrieved_types = []

    for idx in I[0]:

        case_type = df.iloc[idx]["case_type"]

        retrieved_types.append(case_type)

    dominant = Counter(
        retrieved_types
    ).most_common(1)[0][0]

    return dominant


def smart_retrieval(query, top_k=5):

    # Query embedding
    query_embedding = model.encode([query])

    # Stage 1: global retrieval
    D, I = index.search(
        np.array(query_embedding).astype('float32'),
        k=top_k
    )

    # Detect dominant case type
    dominant_type = get_dominant_case_type(I)

    # Filter dataset
    filtered_df = df[
        df["case_type"] == dominant_type
    ].reset_index(drop=True)

    # Create filtered embeddings
    filtered_texts = filtered_df[
        "combined_text"
    ].tolist()

    filtered_embeddings = model.encode(
        filtered_texts
    )

    # Temporary FAISS index
    temp_index = faiss.IndexFlatL2(
        filtered_embeddings.shape[1]
    )

    temp_index.add(
        np.array(filtered_embeddings).astype('float32')
    )

    # Stage 2: refined retrieval
    D2, I2 = temp_index.search(
        np.array(query_embedding).astype('float32'),
        k=min(top_k, len(filtered_df))
    )

    return filtered_df, D2, I2, dominant_type

In [52]:
from collections import defaultdict, Counter

def dataset_evidence_law_mapping(
    query,
    top_k=5
):

    # Smart retrieval
    filtered_df, D, I, dominant_type = smart_retrieval(
        query,
        top_k=top_k
    )

    # Evidence → laws mapping
    evidence_map = defaultdict(list)

    # Process retrieved cases
    for idx in I[0]:

        evidence_text = str(
            filtered_df.iloc[idx]["evidence_types"]
        )

        legal_text = str(
            filtered_df.iloc[idx]["legal_sections"]
        )

        # Split evidence
        evidence_list = evidence_text.split(',')

        # Split laws
        law_list = legal_text.split(';')

        cleaned_laws = []

        # Clean law text
        for law in law_list:

            law = law.strip()

            if law and law.lower() != "nan":

                cleaned_laws.append(law)

        # Map evidence → laws
        for ev in evidence_list:

            ev = ev.strip()

            if not ev:
                continue

            evidence_map[ev].extend(
                cleaned_laws
            )

    # Final structured output
    results = []

    for evidence, laws in evidence_map.items():

        # Count law frequency
        law_counts = Counter(laws)

        filtered_laws = []

        # Keep only repeated/common laws
        for law, count in law_counts.items():

            if count >= 2:

                filtered_laws.append(law)

        # Skip weak mappings
        if len(filtered_laws) == 0:
            continue

        results.append({

            "evidence": evidence,

            "laws": filtered_laws
        })

    return results

In [53]:
# Clean case types
case_types = df["case_type"].dropna().astype(str).unique().tolist()

# Create embeddings
case_type_embeddings = model.encode(
    case_types
)

In [54]:
from sklearn.metrics.pairwise import cosine_similarity
def semantic_case_type_detection(query):

    # Query embedding
    query_embedding = model.encode([query])

    # Similarities
    similarities = cosine_similarity(
        query_embedding,
        case_type_embeddings
    )[0]

    # Best match
    best_idx = similarities.argmax()

    detected_type = case_types[best_idx]

    return detected_type

In [55]:
from collections import Counter

In [56]:
def get_dominant_case_type(I):

    retrieved_types = []

    for idx in I[0]:

        case_type = df.iloc[idx]["case_type"]

        retrieved_types.append(case_type)

    # Count most common type
    dominant = Counter(
        retrieved_types
    ).most_common(1)[0][0]

    return dominant

In [57]:
def smart_retrieval(query, top_k=5):

    # Query embedding
    query_embedding = model.encode([query])

    # STAGE 1:
    # Global retrieval
    D, I = index.search(
        np.array(query_embedding).astype('float32'),
        k=top_k
    )

    # Detect dominant case type
    dominant_type = get_dominant_case_type(I)

    # Filter dataset
    filtered_df = df[
        df["case_type"] == dominant_type
    ].reset_index(drop=True)

    # Create filtered embeddings
    filtered_texts = filtered_df[
        "combined_text"
    ].tolist()

    filtered_embeddings = model.encode(
        filtered_texts
    )

    # Temporary FAISS index
    temp_index = faiss.IndexFlatL2(
        filtered_embeddings.shape[1]
    )

    temp_index.add(
        np.array(filtered_embeddings).astype('float32')
    )

    # STAGE 2:
    # Refined retrieval
    D2, I2 = temp_index.search(
        np.array(query_embedding).astype('float32'),
        k=min(top_k, len(filtered_df))
    )

    return filtered_df, D2, I2, dominant_type

In [58]:

print("\nEvidence-to-Law Mapping:\n")

for item in results:

    # Safety check
    if "laws" not in item:
        continue

    print(f"Evidence: {item['evidence']}")

    print("Related Laws:")

    for law in item["laws"]:

        print(f"   - {law}")

    print("-" * 40)


query = """
The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal.

"""

results = dataset_evidence_law_mapping(query)


Evidence-to-Law Mapping:



judgement


In [59]:
from collections import Counter

def judgment_pattern_analytics(
    query,
    top_k=5
):

    # Smart retrieval
    filtered_df, D, I, dominant_type = smart_retrieval(
        query,
        top_k=top_k
    )

    outcome_list = []

    relief_list = []

    # Collect analytics data
    for idx in I[0]:

        # Judgment outcome
        outcome = str(
            filtered_df.iloc[idx]["judgment_outcome"]
        ).strip()

        if outcome and outcome.lower() != "nan":

            outcome_list.append(outcome)

        # Interim relief / custody
        relief = str(
            filtered_df.iloc[idx][
                "interim_relief_or_custody_status"
            ]
        ).strip()

        if relief and relief.lower() != "nan":

            relief_list.append(relief)

    # Count outcomes
    outcome_counts = Counter(outcome_list)

    # Count reliefs
    relief_counts = Counter(relief_list)

    # Total counts
    total_outcomes = sum(
        outcome_counts.values()
    )

    total_reliefs = sum(
        relief_counts.values()
    )

    outcome_results = []

    relief_results = []

    # Outcome percentages
    for outcome, count in outcome_counts.items():

        percentage = round(
            (count / total_outcomes) * 100,
            2
        )

        outcome_results.append({

            "outcome": outcome,

            "percentage": percentage
        })

    # Relief percentages
    for relief, count in relief_counts.items():

        percentage = round(
            (count / total_reliefs) * 100,
            2
        )

        relief_results.append({

            "relief": relief,

            "percentage": percentage
        })

    # Sort descending
    outcome_results.sort(
        key=lambda x: x["percentage"],
        reverse=True
    )

    relief_results.sort(
        key=lambda x: x["percentage"],
        reverse=True
    )

    return {

        "case_type": dominant_type,

        "outcome_analytics": outcome_results,

        "relief_analytics": relief_results
    }

In [60]:
query = """
The criminal appeal arose from conviction of the appellant under murder-related offences. According to the prosecution, the accused was allegedly involved in a violent incident resulting in the death of the victim. The prosecution relied upon eyewitness testimony, medical evidence, investigation records, and surrounding circumstances to establish guilt. The defence challenged the credibility of witnesses, consistency of prosecution evidence, and legality of conviction. The High Court examined whether the prosecution had successfully proved the offence beyond reasonable doubt and whether the conviction recorded by the trial court required interference in appeal.
"""

analytics_results = judgment_pattern_analytics(query)


In [61]:
print("\nJudgment Pattern Analytics:\n")

print(
    f"Case Type: {analytics_results['case_type']}"
)

# Outcome Trends
print("\nOutcome Trends:\n")

for item in analytics_results["outcome_analytics"]:

    print(
        f"- {item['outcome']} "
        f"→ {item['percentage']}%"
    )

# Relief Trends
print("\nInterim Relief / Custody Trends:\n")

for item in analytics_results["relief_analytics"]:

    print(
        f"- {item['relief']} "
        f"→ {item['percentage']}%"
    )


Judgment Pattern Analytics:

Case Type: Criminal

Outcome Trends:

- Guilty → 100.0%

Interim Relief / Custody Trends:

- Accused remained convicted during appeal → 100.0%


In [62]:
import joblib

joblib.dump(index, "faiss_index.pkl")
joblib.dump(df, "legal_dataset.pkl")

print("Saved successfully")

Saved successfully
